# Zgodność anotatorów: Kappa Cohena i Fleissa

Eksporty z Label Studio (format JSON) zapisz jako `annotation/<iteracja>/export_<imię>.json`, np. `annotation/sample1/export_mikolaj.json`.

In [1]:
import json
import re
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import cohen_kappa_score, confusion_matrix
from statsmodels.stats.inter_rater import aggregate_raters, fleiss_kappa

ANNOT = Path("../annotation")
ANNOTATORS = ["mikolaj", "dominik", "piotrek"]
SENTIMENT = "sentiment"  # from_name z label_config.xml
SPANS = "spans"


In [2]:
def load_export(path: Path) -> dict[int, dict]:
    out = {}
    for task in json.loads(path.read_text(encoding="utf-8")):
        anns = [a for a in task.get("annotations", []) if not a.get("was_cancelled")]
        if not anns:
            continue
        sentiment, spans = None, []
        for res in anns[-1]["result"]:
            if res["from_name"] == SENTIMENT:
                sentiment = res["value"]["choices"][0]
            elif res["from_name"] == SPANS:
                v = res["value"]
                spans.append((v["start"], v["end"], v["labels"][0]))
        out[task["data"]["id"]] = {
            "text": task["data"]["text"],
            "sentiment": sentiment,
            "spans": spans,
        }
    return out


TOKEN = re.compile(r"\w+|[^\w\s]")


def token_labels(text: str, spans: list) -> list[str]:
    toks = list(TOKEN.finditer(text))
    labels = ["O"] * len(toks)
    for start, end, label in sorted(spans):
        for i, m in enumerate(toks):
            if m.start() < end and m.end() > start and labels[i] == "O":
                labels[i] = label
    return labels


In [ ]:
def cohen_pairs(df: pd.DataFrame) -> dict[str, float]:
    return {
        f"{a}-{b}": cohen_kappa_score(df[a], df[b])
        for a, b in combinations(df.columns, 2)
    }


def fleiss(df: pd.DataFrame) -> float:
    table, _ = aggregate_raters(df.to_numpy())
    return fleiss_kappa(table, method="fleiss")


def summarize(text_df: pd.DataFrame, tok_df: pd.DataFrame) -> pd.DataFrame:
    active = tok_df[(tok_df != "O").any(axis=1)]  # tokeny zaznaczone przez min. 1 osobę
    rows = {}
    for name, d in [
        ("cały tekst", text_df),
        ("fragmenty: wszystkie tokeny", tok_df),
        ("fragmenty: tokeny zaznaczone", active),
    ]:
        pairs = cohen_pairs(d)
        rows[name] = {
            **pairs,
            "Cohen (średnia)": np.mean(list(pairs.values())),
            "Fleiss": fleiss(d),
        }
    return pd.DataFrame(rows).T


def evaluate(iteration: str):
    data = {a: load_export(ANNOT / iteration / f"export_{a}.json") for a in ANNOTATORS}
    ids = sorted(set.intersection(*(set(d) for d in data.values())))
    text_df = pd.DataFrame(
        {a: [data[a][i]["sentiment"] for i in ids] for a in ANNOTATORS}, index=ids
    )
    tok = {a: [] for a in ANNOTATORS}
    for i in ids:
        for a in ANNOTATORS:
            tok[a] += token_labels(data[a][i]["text"], data[a][i]["spans"])
    tok_df = pd.DataFrame(tok)
    return data, text_df, tok_df


In [ ]:
ITERATION = "sample1"  # zmień na "sample2" dla drugiej iteracji

data, text_df, tok_df = evaluate(ITERATION)
print(f"{len(text_df)} tekstów, {len(tok_df)} tokenów")
summary = summarize(text_df, tok_df)
summary.round(3)


In [ ]:
# macierze pomyłek dla par anotatorów (cały tekst)
labels = ["plus_m", "minus_m", "zero", "amb"]
for a, b in combinations(ANNOTATORS, 2):
    cm = pd.DataFrame(
        confusion_matrix(text_df[a], text_df[b], labels=labels),
        index=[f"{a}: {l}" for l in labels],
        columns=[f"{b}: {l}" for l in labels],
    )
    display(cm)

# teksty z rozbieżnościami do omówienia
disagree = text_df[text_df.nunique(axis=1) > 1].copy()
disagree["text"] = [data[ANNOTATORS[0]][i]["text"][:300] for i in disagree.index]
print(f"{len(disagree)} / {len(text_df)} tekstów z rozbieżnością")
disagree


In [ ]:
# zgodność z oryginalną anotacją PolEmo (plik gold_*.json jest tylko lokalnie, poza repo)
gold_path = ANNOT / ITERATION / f"gold_{ITERATION}.json"
if gold_path.exists():
    gold = {
        int(k): v.removeprefix("meta_")
        for k, v in json.loads(gold_path.read_text()).items()
    }
    g = pd.Series(gold).loc[text_df.index]
    gold_kappa = {a: cohen_kappa_score(text_df[a], g) for a in ANNOTATORS}
    print("Cohen vs gold:", {a: round(k, 3) for a, k in gold_kappa.items()})
else:
    print("brak pliku gold:", gold_path)


In [ ]:
# porównanie iteracji (pomija te, dla których brakuje eksportów)
results = {}
for it in ["sample1", "sample2"]:
    try:
        _, t, k = evaluate(it)
    except FileNotFoundError:
        continue
    s = summarize(t, k)
    results[it] = s[["Cohen (średnia)", "Fleiss"]]

comparison = pd.concat(results, axis=1) if results else pd.DataFrame()
comparison.round(3)
